In [0]:
import os, sys
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)
from src import config_loader as cl
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"))
log = cl.meta_table(cfg, "run_log")
wm  = cl.meta_table(cfg, "watermark")

In [0]:
# Which runs exist? Find the first incremental run's run_id
display(spark.sql(f"""
  SELECT run_id, pipeline_name, min(started_at) AS first_activity, count(*) AS log_rows
  FROM {log} GROUP BY run_id, pipeline_name ORDER BY first_activity
"""))

In [0]:
# Paste the run_id of your first incremental run (pipeline_name = manual_incremental, earliest one)
RUN_ID = "20261127T073840_fb7cf535"

display(spark.sql(f"""
  SELECT stage, resource_type, load_date, status, pages_fetched, records_read,
         records_written, truncated, started_at, ended_at, data_saved_at
  FROM {log} WHERE run_id = '{RUN_ID}'
  ORDER BY started_at
"""))

In [0]:
# The watermark's own change history: one MERGE per resource per day loaded
display(spark.sql(f"DESCRIBE HISTORY {wm}").select("version", "timestamp", "operation", "operationMetrics"))